# Lecture 7 — Gradients & Barren Plateaus
## UZH Quantum Machine Learning Course — Coding Exercises

Use this notebook together with the PDF exercise sheet.
Requires: `pennylane >= 0.44`, `matplotlib`, `scipy`

Run the first cell to install if needed.

In [ ]:
# Uncomment to install:
# !pip install "pennylane>=0.44" matplotlib scipy

import pennylane.numpy as np   # PennyLane's numpy — supports requires_grad
import pennylane as qml
import matplotlib.pyplot as plt
import time

print("PennyLane:", qml.__version__)

## Exercise B3 — Barren Plateau Demo

We will measure how gradient variance shrinks as qubit count grows.
A barren plateau is present when `Var[∂L/∂θ] ∝ 2^{-n}`.

**Tasks:**
- (a) Complete `make_ansatz` (hardware-efficient: RY+RZ per qubit + CNOT chain per layer)
- (b) Run the variance experiment and plot log₂(variance) vs n_qubits
- (c) Modify to use a local cost function and compare

### B3a: Global cost function with proper entanglement (ring topology)

In [ ]:
def make_ansatz(n_qubits, n_layers):
    """Hardware-efficient ansatz: RY+RZ per qubit, CNOT chain, repeated n_layers times."""
    dev = qml.device("default.qubit", wires=n_qubits)

    # We use diff_method='backprop' here so this simulation experiment
    # runs quickly.  On real quantum hardware you would use 'parameter-shift'.
    @qml.qnode(dev, diff_method="backprop")
    def circuit(params):
        # TODO: for l in range(n_layers):
        #   apply RY(params[l, q]) on each qubit q
        #   apply RZ(params[l, q + n_qubits]) on each qubit q
        #   apply CNOT chain with closing the ring: qubit 0->1->2->...->n_qubits-1
        # YOUR CODE HERE
        raise NotImplementedError
        return qml.expval(qml.PauliZ(0))  # global cost

    return circuit


qubit_counts = [2, 4, 6, 8, 10]
n_samples = 50   # 50 samples per qubit count is enough for variance estimation

variances = []
n_layers_list = []  # stores depth used per n -- needed again in B3b
for n in qubit_counts:
    n_layers_bp = n  # depth proportional to width so the ansatz forms a 2-design
    n_layers_list.append(n_layers_bp)
    circ = make_ansatz(n, n_layers_bp)
    grads = []
    for _ in range(n_samples):
        params = np.array(np.random.uniform(0, 2 * np.pi, (n_layers_bp, 2 * n)),
                          requires_grad=True)
        # TODO: compute gradient using qml.grad(circ)(params)
        # Extract a buried middle-circuit parameter to see proper BP scaling:
        #   g = qml.grad(circ)(params)
        #   grads.append(float(g[n_layers_bp // 2, n // 2]))
        # YOUR CODE HERE
        raise NotImplementedError
    variances.append(float(np.var(grads)))

log_var = np.log2(np.array(variances))
fit = np.polyfit(qubit_counts, log_var, 1)

# TODO: plot log_var vs qubit_counts with fit line
# YOUR CODE HERE
print(f"Slope: {fit[0]:.3f}  -- negative slope confirms exponential suppression = barren plateau")


### B3b — Local Cost Function

Replace the global `qml.expval(qml.PauliZ(0))` with a local cost that averages Z over all qubits.
A local cost avoids the concentration-of-measure effect and has only polynomial gradient suppression.

In [ ]:
def make_ansatz_local(n_qubits, n_layers):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, diff_method="backprop")
    def circuit(params):
        # TODO: same ansatz as above (RY+RZ per qubit, CNOT chain)
        # YOUR CODE HERE
        raise NotImplementedError
        # TODO: return LOCAL cost -- Z on the MIDDLE qubit only.
        # Hint: qml.expval(qml.PauliZ(n_qubits // 2))
        # The buried parameter at (n//2, n//2) sits directly in the light cone
        # of this observable but is far from Z(0) -- that is the key contrast.
        raise NotImplementedError

    return circuit


variances_local = []
# Reuse n_layers_list and n_samples from B3a above
for n, n_layers_bp in zip(qubit_counts, n_layers_list):
    circ = make_ansatz_local(n, n_layers_bp)
    grads = []
    for _ in range(n_samples):
        params = np.array(np.random.uniform(0, 2 * np.pi, (n_layers_bp, 2 * n)),
                          requires_grad=True)
        # TODO: compute gradient, extract same buried parameter g[n_layers_bp // 2, n // 2]
        # YOUR CODE HERE
        raise NotImplementedError
    variances_local.append(float(np.var(grads)))

log_var_local = np.log2(np.array(variances_local))
fit_local = np.polyfit(qubit_counts, log_var_local, 1)

# TODO: plot both global and local curves on the same figure, print slopes
# YOUR CODE HERE
print(f"Global slope: {fit[0]:.3f}  |  Local slope: {fit_local[0]:.3f}")
print("-> Local cost (Z on middle qubit) has shallower decay:")
print("   buried parameter is in the light cone of Z(n/2) but far from Z(0).")


## Exercise C1 — Identity-Block Initialisation

Random initialisation puts the circuit near a random unitary → barren plateau.
**Identity-block initialisation** starts all rotation angles near zero, keeping the circuit close to the identity map. Near the identity, each qubit sees a nearly separable state, and gradients are O(1).

In [ ]:
def identity_init(n_qubits, n_layers, noise_scale=0.01):
    """Return params of shape (n_layers, 2*n_qubits) near zero + small Gaussian noise."""
    # TODO: np.array of shape (n_layers, 2*n_qubits), values ~ Normal(0, noise_scale),
    #       requires_grad=True
    # YOUR CODE HERE
    raise NotImplementedError


# Use depth ∝ width, consistent with B3, so both inits face the same circuit.
n, L = 8, 8
circ = make_ansatz(n, L)


def collect_grads(init_fn, n_samples=50):
    """Return list of grad[0,0] values (surface parameter) over n_samples initialisations."""
    grads = []
    for _ in range(n_samples):
        params = init_fn()
        # TODO: compute gradient with qml.grad(circ)(params), append float(g[0, 0])
        # YOUR CODE HERE
        raise NotImplementedError
    return grads


grads_random   = collect_grads(lambda: np.array(
    np.random.uniform(0, 2 * np.pi, (L, 2 * n)), requires_grad=True))
grads_identity = collect_grads(lambda: identity_init(n, L))

mean_abs_random   = float(np.mean(np.abs(grads_random)))
mean_abs_identity = float(np.mean(np.abs(grads_identity)))
print(f"Random init    -- Var: {np.var(grads_random):.4f}  |  Mean |grad|: {mean_abs_random:.4f}")
print(f"Identity init  -- Var: {np.var(grads_identity):.6f}  |  Mean |grad|: {mean_abs_identity:.6f}")
print()
print("Interpretation:")
print("  Random init   -> large, unpredictable gradients (high variance across runs).")
print("  Identity init -> small, CONSISTENT gradients (low variance -- every run is similar).")
print("  At n=8 random init is not yet in severe BP, so its gradients are large.")
print("  At n >> 10, random init gradients would vanish exponentially (slope from B3).")
print("  Identity init gradients stay bounded below by the noise scale regardless of n.")

# TODO: plot two histograms side by side: Random init vs Identity-block init
# Use sharey=False so axes scale independently; mark x=0 with a dashed line.
# YOUR CODE HERE


## Exercise C2 — Rotosolve: Analytic Single-Parameter Optimisation

For rotation gates, the loss is exactly `L(θ) = A·sin(θ) + B·cos(θ) + C`.
Three circuit evaluations (at θ=0, π/2, -π/2) determine A, B, C analytically,
giving the exact global minimum. Cycling over all parameters = **Rotosolve**.

In [ ]:
def rotosolve_step(circuit_fn, params, k):
    """
    Analytically minimise circuit_fn w.r.t. the k-th flat parameter.
    Uses 3 evaluations: f(0), f(pi/2), f(-pi/2).
    """
    params_flat = np.array(params).flatten().copy()

    def f(t):
        p = params_flat.copy()
        p[k] = t
        return float(circuit_fn(p.reshape(params.shape)))

    # TODO: compute C = f(0), A = (f(pi/2) - f(-pi/2)) / 2, B = (f(pi/2) + f(-pi/2)) / 2 - C
    # TODO: set params_flat[k] = atan2(-A, -B) + pi
    # YOUR CODE HERE
    raise NotImplementedError

    return np.array(params_flat.reshape(params.shape), requires_grad=True)


n, L = 4, 2   # small circuit -- 20 cycles is more than enough for convergence
circ = make_ansatz(n, L)

# Start from all-zeros: circuit = identity → Z(0) = +1.0 (worst case for minimisation).
# Rotosolve should converge quickly to the global minimum near -1.0.
params = np.array(np.zeros((L, 2 * n)), requires_grad=True)
print(f"Initial loss (all-zeros params): {float(circ(params)):.4f}  (expect +1.0)")

losses_rotosolve = []
for cycle in range(20):
    for k in range(params.size):
        # TODO: apply rotosolve_step for each parameter k
        # YOUR CODE HERE
        pass
    losses_rotosolve.append(float(circ(params)))

# TODO: plot losses_rotosolve vs cycle with a dashed horizontal line at -1.0
# print initial (+1.0) and final loss
# YOUR CODE HERE


## Mini-Project — VQE Barren Plateau Benchmark

**Task**: compare 3 training strategies on the 6-qubit transverse-field Ising model.

Hamiltonian: `H = Σ Z_i Z_{i+1} + Σ X_i`

| Strategy | Init | Cost |
|---|---|---|
| 1 — Baseline | Random | Global H |
| 2 — Identity init | Identity-block | Global H |
| 3 — Local cost | Identity-block | ZZ terms only |

Report: loss curve (200 steps), final energy, gradient variance at init.
See `lecture7_solutions.ipynb` for the complete implementation.

In [ ]:
import scipy.optimize

n_qubits, n_layers = 6, 3

# --- Build Hamiltonian using arithmetic (PL >= 0.40 compatible) ---
def ising_hamiltonian(n):
    H = sum(qml.PauliZ(i) @ qml.PauliZ(i + 1) for i in range(n - 1))
    H = H + sum(qml.PauliX(i) for i in range(n))
    return H

H_global = ising_hamiltonian(n_qubits)

def ising_local(n):
    """ZZ terms only -- local cost for barren plateau mitigation."""
    return sum(qml.PauliZ(i) @ qml.PauliZ(i + 1) for i in range(n - 1))

H_local = ising_local(n_qubits)
dev = qml.device("default.qubit", wires=n_qubits)

def make_vqe(hamiltonian):
    @qml.qnode(dev, diff_method="backprop")
    def circuit(params):
        # TODO: implement hardware-efficient ansatz (n_layers layers)
        # YOUR CODE HERE
        raise NotImplementedError
        return qml.expval(hamiltonian)
    return circuit

# Evaluator on full H -- used at the end to fairly compare all 3 strategies
circ_eval = make_vqe(H_global)

n_steps = 100
# TODO: run n_steps Adam steps for each of the 3 strategies and plot convergence
#
# Strategies:
#   1 -- Random init,   train on H_global
#   2 -- Identity init, train on H_global
#   3 -- Identity init, train on H_local
#
# Hint: opt = qml.AdamOptimizer(stepsize=0.05)
#       params, loss = opt.step_and_cost(circ, params)
#
# After training, evaluate every strategy's final params on H_global (circ_eval)
# for a fair comparison -- strategy 3 trains on a different observable.
# YOUR CODE HERE


## Bonus — PennyLane Catalyst: JIT-compiled Gradients

PennyLane Catalyst (`@qml.qjit`) compiles the entire forward + backward pass to native code via MLIR/LLVM.
Instead of Python re-entering the circuit on every call, the compiled function runs directly — typically **5–50× faster** on CPU.

**Key differences from standard PennyLane:**
- Device must be `lightning.qubit` (not `default.qubit`)
- Use plain `numpy` (not `pennylane.numpy` with `requires_grad`)
- Use `catalyst.grad` instead of `qml.grad`

Requires: `pip install pennylane-catalyst`

In [ ]:
# Uncomment to install:
# !pip install pennylane-catalyst

import numpy as np_plain   # plain numpy -- required by Catalyst (NOT pennylane.numpy)
import pennylane.numpy as pnp  # pennylane numpy for standard path
import pennylane as qml
import time

n_wires = 6
N_REPS = 50  # number of gradient calls to time

# Standard PennyLane (baseline)
dev_std = qml.device("default.qubit", wires=n_wires)

@qml.qnode(dev_std, diff_method="parameter-shift")
def circuit_std(params):
    for i in range(n_wires):
        qml.RY(params[i], wires=i)
    for i in range(n_wires - 1):
        qml.CNOT(wires=[i, i + 1])
    return qml.expval(qml.PauliZ(0))

# TODO: time N_REPS gradient evaluations using qml.grad(circuit_std)
# Use pnp.array(..., requires_grad=True) for the pennylane numpy path
# Warm up with one call before timing, then time N_REPS calls
# Print: f"Standard PennyLane: {t_std:.3f} s  ({t_std/N_REPS*1e3:.2f} ms/call)"
# YOUR CODE HERE
raise NotImplementedError

# Catalyst JIT (fast)
try:
    import catalyst

    dev_jit = qml.device("lightning.qubit", wires=n_wires)

    @qml.qjit
    @qml.qnode(dev_jit)
    def circuit_jit(params):
        for i in range(n_wires):
            qml.RY(params[i], wires=i)
        for i in range(n_wires - 1):
            qml.CNOT(wires=[i, i + 1])
        return qml.expval(qml.PauliZ(0))

    # TODO: time N_REPS gradient evaluations using catalyst.grad(circuit_jit)
    # Use np_plain array (plain numpy, not pennylane.numpy)
    # Warm up with one call before timing (first call triggers JIT compilation)
    # Print timing and speedup, plot bar chart
    # YOUR CODE HERE
    raise NotImplementedError

except ImportError:
    print("pennylane-catalyst not installed.")
    print("Install with:  pip install pennylane-catalyst")
    print("Then re-run this cell.")
